# E-commerce Sales & Customer Analysis

**Portfolio project | Python · Pandas · Matplotlib · Seaborn · RFM analysis**

## Business objective
Analyse historical transactions from a UK-based online retailer to understand revenue trends, product performance, geographic performance, and customer value.

## Dataset
UCI Machine Learning Repository — [Online Retail](https://archive.ics.uci.edu/dataset/352/online-retail), 541,909 transaction-line records covering 1 December 2010 to 9 December 2011.

**Attribution:** Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33. Licensed under CC BY 4.0.

> Important: This notebook is designed to run end-to-end after downloading the Excel file or fetching it with `ucimlrepo`. The dataset could not be downloaded into this chat's execution environment, so no results are fabricated. Run the notebook to calculate the real figures.


## 1. Setup and load data

In [ ]:
# If needed, install dependencies in your environment:
# %pip install pandas numpy matplotlib seaborn openpyxl ucimlrepo

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("data/Online Retail.xlsx")

if DATA_PATH.exists():
    raw = pd.read_excel(DATA_PATH)
else:
    # Alternative: fetch directly from the UCI Machine Learning Repository
    from ucimlrepo import fetch_ucirepo
    dataset = fetch_ucirepo(id=352)
    raw = pd.concat(
        [dataset.data.features, dataset.data.targets],
        axis=1
    )

print("Shape:", raw.shape)
display(raw.head())
display(raw.info())


## 2. Data quality assessment

In [ ]:
print("Duplicate rows:", raw.duplicated().sum())
display(raw.isna().sum().sort_values(ascending=False).to_frame("missing_values"))
display(raw.describe(include="all").T)


## 3. Data cleaning and revenue calculation

Cleaning decisions:
- Remove exact duplicate transaction lines.
- Exclude rows without a customer ID from customer-level segmentation (retain a separate copy for sales-level checks if desired).
- Exclude cancellation invoices (invoice numbers beginning with `C`) from completed-sales analysis.
- Exclude non-positive quantities and unit prices from completed-sales revenue analysis.
- Parse invoice dates and calculate line revenue as `Quantity × UnitPrice`.

These rules define a completed-sales view; returns/cancellations are not treated as positive sales.


In [ ]:
df = raw.copy()
df.columns = df.columns.str.strip()

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["InvoiceNo"] = df["InvoiceNo"].astype("string")
df["is_cancellation"] = df["InvoiceNo"].str.upper().str.startswith("C", na=False)

clean = df.drop_duplicates().copy()
clean = clean[
    clean["InvoiceDate"].notna()
    & (~clean["is_cancellation"])
    & (clean["Quantity"] > 0)
    & (clean["UnitPrice"] > 0)
].copy()

clean["Revenue"] = clean["Quantity"] * clean["UnitPrice"]
clean["YearMonth"] = clean["InvoiceDate"].dt.to_period("M").astype(str)
clean["Weekday"] = clean["InvoiceDate"].dt.day_name()

print(f"Raw rows: {len(raw):,}")
print(f"Rows after completed-sales cleaning: {len(clean):,}")
print(f"Rows removed: {len(raw) - len(clean):,}")
print(f"Unique invoices: {clean['InvoiceNo'].nunique():,}")
print(f"Unique customers with ID: {clean['CustomerID'].nunique():,}")
print(f"Total recorded revenue: £{clean['Revenue'].sum():,.2f}")


## 4. Monthly revenue trend

In [ ]:
monthly = clean.groupby("YearMonth", as_index=False).agg(
    Revenue=("Revenue", "sum"),
    Orders=("InvoiceNo", "nunique"),
    Units=("Quantity", "sum")
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly, x="YearMonth", y="Revenue", marker="o")
plt.title("Monthly revenue")
plt.xlabel("Month")
plt.ylabel("Revenue (£)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(monthly.sort_values("Revenue", ascending=False).head(5))


## 5. Product performance

In [ ]:
product_perf = clean.groupby("Description", dropna=True).agg(
    Units_Sold=("Quantity", "sum"),
    Revenue=("Revenue", "sum"),
    Orders=("InvoiceNo", "nunique")
).sort_values("Revenue", ascending=False)

display(product_perf.head(10))

top_products = product_perf.head(10).sort_values("Revenue")
plt.figure(figsize=(10, 6))
sns.barplot(data=top_products.reset_index(), x="Revenue", y="Description")
plt.title("Top 10 products by revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 6. Country performance

In [ ]:
country_perf = clean.groupby("Country").agg(
    Revenue=("Revenue", "sum"),
    Orders=("InvoiceNo", "nunique"),
    Customers=("CustomerID", "nunique")
).sort_values("Revenue", ascending=False)

display(country_perf.head(10))

top_countries = country_perf.head(10).sort_values("Revenue")
plt.figure(figsize=(10, 6))
sns.barplot(data=top_countries.reset_index(), x="Revenue", y="Country")
plt.title("Top 10 countries by revenue")
plt.xlabel("Revenue (£)")
plt.ylabel("")
plt.tight_layout()
plt.show()


## 7. Customer analysis: RFM segmentation

RFM means:
- **Recency:** days since the customer's most recent purchase (lower is better).
- **Frequency:** number of distinct invoices (higher is better).
- **Monetary:** total recorded revenue (higher is better).

The reference date is set to one day after the latest invoice date in the dataset, making the calculation reproducible for this historical dataset.


In [ ]:
customer_data = clean.dropna(subset=["CustomerID"]).copy()
reference_date = customer_data["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = customer_data.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (reference_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum")
)

# Rank customers into quintiles. Recency is reversed: lower recency gets a higher score.
rfm["R"] = pd.qcut(rfm["Recency"].rank(method="first"), 5, labels=[5,4,3,2,1]).astype(int)
rfm["F"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["M"] = pd.qcut(rfm["Monetary"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["RFM_Score"] = rfm[["R", "F", "M"]].sum(axis=1)

def segment_customer(row):
    if row["R"] >= 4 and row["F"] >= 4 and row["M"] >= 4:
        return "Best customers"
    if row["R"] >= 3 and row["F"] >= 3:
        return "Loyal / engaged"
    if row["R"] <= 2 and row["F"] >= 3:
        return "At risk"
    if row["F"] <= 2:
        return "New / occasional"
    return "Potential loyalists"

rfm["Segment"] = rfm.apply(segment_customer, axis=1)
display(rfm.head())
display(rfm.groupby("Segment").agg(
    Customers=("Segment", "size"),
    Average_Recency=("Recency", "mean"),
    Average_Frequency=("Frequency", "mean"),
    Total_Monetary=("Monetary", "sum")
).sort_values("Customers", ascending=False))


## 8. Key performance indicators

In [ ]:
kpis = {
    "Total revenue (£)": clean["Revenue"].sum(),
    "Unique orders": clean["InvoiceNo"].nunique(),
    "Unique identified customers": clean["CustomerID"].nunique(),
    "Average order value (£)": clean.groupby("InvoiceNo")["Revenue"].sum().mean(),
    "Average revenue per identified customer (£)": (
        customer_data.groupby("CustomerID")["Revenue"].sum().mean()
    ),
}
display(pd.Series(kpis, name="Value").to_frame())


## 9. Findings and recommendations — complete after running

Use the calculated outputs above to write evidence-based conclusions. Do not fill this section with assumptions.

Suggested questions:
1. Which month had the highest revenue, and what might explain the peak?
2. Is revenue concentrated in a small number of products or countries?
3. How large is the best-customer segment?
4. Which customers appear at risk based on recency?
5. What practical action should the retailer take?

## Limitations
- This is historical data from one retailer, not a current market benchmark.
- Revenue is calculated from transaction line quantity multiplied by unit price; it is not necessarily net profit.
- The completed-sales view excludes cancellations and non-positive quantity/price rows. Returns are therefore not analysed as a separate process here.
- RFM segment labels are rule-based illustrative groupings, not ground-truth customer labels.
